# Hidden Poetic Schools — One File, Run Top to Bottom

This single notebook does everything: scrapes the poems, embeds them,
clusters them, shows you the result. No other files needed.

Click the first cell below, then press **Shift+Enter** repeatedly to run
each cell in order. Don't skip any.


In [1]:
# CELL 1 — Install packages (only takes a while the first time)
!pip -q install sentence-transformers torch scikit-learn umap-learn hdbscan pandas numpy matplotlib seaborn requests beautifulsoup4 lxml tqdm pillow


In [2]:
# CELL 2 — Settings (edit these if you want, otherwise leave as-is)
from pathlib import Path

DB_PATH = Path("poems.db")
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42
SBERT_MODEL_NAME = "akhooli/Arabic-SBERT-100K"
MIN_POEMS_PER_POET = 2
MIN_VERSES_PER_POEM = 2

UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
UMAP_N_COMPONENTS_CLUSTER = 10
HDBSCAN_MIN_CLUSTER_SIZE = 5

ALDIWAN_BASE_URL = "https://www.aldiwan.net"
ALDIWAN_CATEGORY_URL = f"{ALDIWAN_BASE_URL}/cat-poets-pre-islamic-period"
SCRAPER_DELAY_SECONDS = 1.5

import random
import numpy as np
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Settings loaded.")


Settings loaded.


In [3]:
# CELL 3 — Scrape the poems from aldiwan.net into poems.db
# SLOW: can take a couple hours for the full corpus. Safe to re-run if it
# gets interrupted — it skips poets already saved.
import re
import sqlite3
import time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from tqdm import tqdm


def init_db(db_path):
    conn = sqlite3.connect(db_path)
    conn.executescript("""
        CREATE TABLE IF NOT EXISTS poets (
            poet_id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL, slug TEXT UNIQUE NOT NULL, bio TEXT
        );
        CREATE TABLE IF NOT EXISTS poems (
            poem_id INTEGER PRIMARY KEY AUTOINCREMENT,
            poet_id INTEGER NOT NULL, title TEXT,
            source_url TEXT UNIQUE NOT NULL, n_verses INTEGER
        );
        CREATE TABLE IF NOT EXISTS verses (
            verse_id INTEGER PRIMARY KEY AUTOINCREMENT,
            poem_id INTEGER NOT NULL, verse_order INTEGER, text TEXT
        );
    """)
    conn.commit()
    return conn


def is_valid_verse(text, min_chars=3):
    if not text or len(text.strip()) < min_chars:
        return False
    return len(re.findall(r"[\u0600-\u06FF]", text)) >= min_chars


session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})


def get_soup(url):
    resp = session.get(url, timeout=20)
    resp.raise_for_status()
    time.sleep(SCRAPER_DELAY_SECONDS)
    return BeautifulSoup(resp.text, "lxml")


def scrape_all(db_path, limit_poets=None):
    conn = init_db(db_path)
    soup = get_soup(ALDIWAN_CATEGORY_URL)

    poet_links, seen = [], set()
    for a in soup.select("a[href*='/cat-poet-']"):
        href = a.get("href", "")
        if "/cat-poet-" not in href:
            continue
        url = urljoin(ALDIWAN_BASE_URL, href)
        name = a.get_text(strip=True)
        if name and url not in seen:
            seen.add(url)
            poet_links.append({"name": name, "url": url})

    if limit_poets:
        poet_links = poet_links[:limit_poets]
    print(f"Found {len(poet_links)} poets to scrape.")

    for poet in tqdm(poet_links, desc="Poets"):
        slug = poet["url"].rstrip("/").split("/")[-1]
        if conn.execute("SELECT 1 FROM poets WHERE slug=?", (slug,)).fetchone():
            continue
        try:
            poet_soup = get_soup(poet["url"])
        except requests.RequestException:
            continue

        bio_tag = poet_soup.find("h4")
        bio = bio_tag.get_text(strip=True) if bio_tag else ""
        conn.execute("INSERT OR IGNORE INTO poets (name, slug, bio) VALUES (?,?,?)",
                     (poet["name"], slug, bio))
        conn.commit()
        poet_id = conn.execute("SELECT poet_id FROM poets WHERE slug=?", (slug,)).fetchone()[0]

        poem_links, pseen = [], set()
        for a in poet_soup.select("a[href*='/poem']"):
            href = a.get("href", "")
            if not re.search(r"/poem(\d+)\.html", href):
                continue
            url = urljoin(ALDIWAN_BASE_URL, href)
            title = a.get_text(strip=True)
            if title and url not in pseen:
                pseen.add(url)
                poem_links.append({"title": title, "url": url})

        for poem in poem_links:
            if conn.execute("SELECT 1 FROM poems WHERE source_url=?", (poem["url"],)).fetchone():
                continue
            try:
                psoup = get_soup(poem["url"])
            except requests.RequestException:
                continue
            verses = [t.get_text(strip=True) for t in psoup.select("h3")]
            verses = [v for v in verses if is_valid_verse(v)]
            if len(verses) < MIN_VERSES_PER_POEM:
                continue
            cur = conn.execute(
                "INSERT INTO poems (poet_id, title, source_url, n_verses) VALUES (?,?,?,?)",
                (poet_id, poem["title"], poem["url"], len(verses)),
            )
            poem_id = cur.lastrowid
            conn.executemany(
                "INSERT INTO verses (poem_id, verse_order, text) VALUES (?,?,?)",
                [(poem_id, i, v) for i, v in enumerate(verses)],
            )
            conn.commit()

    n_poets = conn.execute("SELECT COUNT(*) FROM poets").fetchone()[0]
    n_poems = conn.execute("SELECT COUNT(*) FROM poems").fetchone()[0]
    print(f"Done. poets={n_poets} poems={n_poems} -> {db_path}")
    conn.close()


# Quick test with 5 poets first. Once you confirm this works, change
# limit_poets=5 to limit_poets=None below and re-run this cell for the
# full scrape.
scrape_all(DB_PATH, limit_poets=5)


Found 0 poets to scrape.


Poets: 0it [00:00, ?it/s]

Done. poets=0 poems=0 -> poems.db


**Check here before continuing:** did the cell above print something
like `Done. poets=5 poems=... -> poems.db`? If yes, go back up, change
`limit_poets=5` to `limit_poets=None`, and re-run that cell to scrape
everyone (this will take a couple hours — you can walk away).


In [4]:
# CELL 4 — Load what we scraped and clean it up
import sqlite3
import pandas as pd


def normalize_arabic(text):
    import re
    text = re.sub(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]", "", text)
    text = re.sub(r"\u0640", "", text)
    text = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", text)
    text = re.sub(r"\u0649", "\u064a", text)
    text = re.sub(r"\u0629", "\u0647", text)
    return re.sub(r"\s+", " ", text).strip()


conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("""
    SELECT p.name AS poet_name, pm.poem_id AS poem_id, v.verse_order, v.text
    FROM verses v
    JOIN poems pm ON v.poem_id = pm.poem_id
    JOIN poets p ON pm.poet_id = p.poet_id
    ORDER BY p.poet_id, pm.poem_id, v.verse_order
""", conn)
conn.close()

df["text"] = df["text"].apply(normalize_arabic)
df = df[df["text"].str.len() > 2]

poems_per_poet = df.groupby("poet_name")["poem_id"].transform("nunique")
df = df[poems_per_poet >= MIN_POEMS_PER_POET]

poems_by_poet = {}
for poet, pdf in df.groupby("poet_name"):
    poems = []
    for pid, one_poem in pdf.groupby("poem_id"):
        poems.append(one_poem.sort_values("verse_order")["text"].tolist())
    poems_by_poet[poet] = poems

print(f"Poets: {len(poems_by_poet)} | Poems: {df['poem_id'].nunique()} | Verses: {len(df)}")


Poets: 0 | Poems: 0 | Verses: 0


In [ ]:
# CELL 5 — Embed every poet with Arabic Sentence-BERT
# (verse -> average per poem -> average per poet)
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(SBERT_MODEL_NAME)

poet_embeddings = {}
for poet, poems in poems_by_poet.items():
    poem_vectors = []
    for verses in poems:
        if not verses:
            continue
        embs = model.encode(verses, show_progress_bar=False, normalize_embeddings=True)
        poem_vectors.append(embs.mean(axis=0))
    if poem_vectors:
        poet_embeddings[poet] = np.mean(poem_vectors, axis=0)

print(f"Embedded {len(poet_embeddings)} poets.")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/637 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/761k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.78M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Embedded 0 poets.


In [5]:
# CELL 6 — Cluster the poets into "schools"
import umap
import hdbscan
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score

poet_names = sorted(poet_embeddings.keys())
emb_matrix = np.stack([poet_embeddings[n] for n in poet_names])

sim_matrix = cosine_similarity(emb_matrix)

reducer = umap.UMAP(n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST,
                     n_components=UMAP_N_COMPONENTS_CLUSTER, metric="cosine",
                     random_state=RANDOM_SEED)
reduced = reducer.fit_transform(emb_matrix)

clusterer = hdbscan.HDBSCAN(min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE)
labels = clusterer.fit_predict(reduced)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
mask = labels != -1
sil = silhouette_score(reduced[mask], labels[mask]) if n_clusters >= 2 else None

print(f"Clusters found: {n_clusters}")
print(f"Silhouette score: {sil}")

# also make a 2D version, just for plotting
reducer_2d = umap.UMAP(n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST,
                        n_components=2, metric="cosine", random_state=RANDOM_SEED)
coords_2d = reducer_2d.fit_transform(emb_matrix)


NameError: name 'poet_embeddings' is not defined

In [ ]:
# CELL 7 — Show the picture
import matplotlib.pyplot as plt
import seaborn as sns

unique_labels = sorted(set(labels))
palette = sns.color_palette("husl", len([l for l in unique_labels if l != -1]))
color_map, idx = {}, 0
for l in unique_labels:
    if l == -1:
        color_map[l] = (0.7, 0.7, 0.7)
    else:
        color_map[l] = palette[idx]
        idx += 1

plt.figure(figsize=(10, 9))
for l in unique_labels:
    sel = labels == l
    label_str = "Noise" if l == -1 else f"School {l}"
    plt.scatter(coords_2d[sel, 0], coords_2d[sel, 1], s=40, alpha=0.85,
                color=color_map[l], label=label_str, edgecolor="k", linewidth=0.3)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.title("Hidden Stylistic Schools of Pre-Islamic Poetry")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "clusters.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# CELL 8 — Save the results to a CSV you can open in Excel
result_df = pd.DataFrame({"poet_name": poet_names, "cluster": labels})
result_df = result_df.sort_values("cluster")
result_df.to_csv("cluster_assignments.csv", index=False)
print("Saved to cluster_assignments.csv")
result_df


## Done

You now have:
- `poems.db` — the scraped corpus
- `figures/clusters.png` — the picture of the schools
- `cluster_assignments.csv` — which poet is in which cluster

That's the whole pipeline, one notebook, no other files needed.
